# High Quality Evidence Retrieval

Goal: retrieve the best top-3 evidence candidates for phase 3 LLM fact-check judging.

This notebook uses a wide-then-rerank design:

1. Build query packs from refined OpenRouter outputs.
2. Retrieve text evidence from dense `text_vector` and sparse keyword search.
3. Retrieve image evidence with original CLIP and fine-tuned CLIP text-to-image search.
4. Fuse candidates with weighted RRF.
5. Rerank to a final top 3 evidence package.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import html
import io
import json
import math
import re
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from PIL import Image
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseVector
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"database", "refined"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

QDRANT_URL = "http://localhost:6333"
COLLECTIONS = {
    "fixed_size": "fixed_size",
    "semantic": "semantic",
}
DEFAULT_COLLECTION = COLLECTIONS["semantic"]

REFINED_DIR = PROJECT_ROOT / "refined" / "refined_outputs_openrouter"
DATASET_ROOT = PROJECT_ROOT / "FinalDataset"

BKVEC_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"
IMG_MODEL = "sentence-transformers/clip-ViT-B-32"
IMG_MODEL_FINETUNED = PROJECT_ROOT / "models" / "clip-vit-b32-finetuned-final-final" / "best"

TEXT_VECTOR = "text_vector"
SPARSE_VECTOR = "sparse"
IMAGE_VECTOR = "image_vector"
IMAGE_VECTOR_FINETUNED = "image_vector_finetuned"

CANDIDATES_PER_BRANCH = 30
FINAL_TOP_K = 3
RRF_K = 60
CROSS_ENCODER_MODEL = "namdp-ptit/ViRanker"
CROSS_ENCODER_MAX_LENGTH = 512
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

client = QdrantClient(url=QDRANT_URL)
print(f"Project root: {PROJECT_ROOT}")
print(f"Refined dir: {REFINED_DIR}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Qdrant collections: {[c.name for c in client.get_collections().collections]}")


## Load Refined Outputs

The notebook accepts per-model CSV files from `refined/refined_outputs_openrouter`. It does not require a combined CSV.

In [ ]:
def _safe_json(value: Any, default: Any):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    if isinstance(value, (dict, list)):
        return value
    text = str(value).strip()
    if not text:
        return default
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return default

def load_refined_outputs(refined_dir: Path = REFINED_DIR) -> pd.DataFrame:
    files = sorted(p for p in refined_dir.glob("refined_*.csv") if "combined" not in p.name)
    if not files:
        raise FileNotFoundError(f"No refined_*.csv files found under {refined_dir}")
    frames = []
    for path in files:
        frame = pd.read_csv(path)
        frame["_source_file"] = path.name
        frames.append(frame)
    out = pd.concat(frames, ignore_index=True)
    if "refine_error" in out.columns:
        out = out[out["refine_error"].fillna("").astype(str).str.strip().eq("")].copy()
    return out

refined = load_refined_outputs()
print(refined.shape)
display(refined[["id", "claim", "model_alias", "refined_primary_retrieval_query", "_source_file"]].head(10))


## Build Query Packs

A query pack keeps text, keyword, and visual queries separate so each retriever gets the query type it is good at.

In [ ]:
def _dedupe_keep_order(items: list[str], max_items: int | None = None) -> list[str]:
    seen = set()
    out = []
    for item in items:
        text = str(item or "").strip()
        key = text.lower()
        if text and key not in seen:
            seen.add(key)
            out.append(text)
    return out[:max_items] if max_items else out

def build_query_pack(row: pd.Series) -> dict[str, Any]:
    search_queries = _safe_json(row.get("refined_search_queries"), {})
    claim_atoms = _safe_json(row.get("refined_claim_atoms"), [])
    visual_observations = _safe_json(row.get("refined_visual_observations"), [])
    retrieval_focus = _safe_json(row.get("refined_retrieval_focus"), {})
    verification_targets = _safe_json(row.get("refined_verification_targets"), [])

    atom_queries = []
    for atom in claim_atoms if isinstance(claim_atoms, list) else []:
        atom_queries.extend(atom.get("retrieval_queries", []) if isinstance(atom, dict) else [])

    observation_queries = []
    for obs in visual_observations if isinstance(visual_observations, list) else []:
        if isinstance(obs, dict):
            observation_queries.append(obs.get("text", ""))
            observation_queries.extend(obs.get("visible_evidence", []))

    text_queries = _dedupe_keep_order([
        row.get("refined_primary_retrieval_query", ""),
        row.get("refined_normalized_claim", ""),
        *search_queries.get("semantic", []),
        *atom_queries,
        *verification_targets,
    ], max_items=10)

    keyword_queries = _dedupe_keep_order([
        *search_queries.get("keywords", []),
        *verification_targets,
    ], max_items=12)

    visual_queries = _dedupe_keep_order([
        *search_queries.get("visual", []),
        *observation_queries,
    ], max_items=10)

    if retrieval_focus.get("cross_modal", False):
        visual_queries = _dedupe_keep_order([*visual_queries, row.get("refined_primary_retrieval_query", "")], max_items=12)

    return {
        "claim_id": row.get("id"),
        "claim": row.get("claim", ""),
        "model_alias": row.get("model_alias", ""),
        "text_queries": text_queries,
        "keyword_queries": keyword_queries,
        "visual_queries": visual_queries,
        "retrieval_focus": retrieval_focus,
    }

def pick_refined_row(claim_id: int | str | None = None, model_alias: str | None = None, index: int = 0) -> pd.Series:
    frame = refined.copy()
    if claim_id is not None:
        frame = frame[frame["id"].astype(str).eq(str(claim_id))]
    if model_alias is not None:
        frame = frame[frame["model_alias"].astype(str).eq(model_alias)]
    if frame.empty:
        raise ValueError("No refined row matches the requested filters")
    return frame.iloc[index]

sample_row = pick_refined_row(index=0)
sample_pack = build_query_pack(sample_row)
sample_pack


## Load Embedding Models

These are the same model families used when building the Qdrant vectors. Query vectors must use the same vector space as the stored named vector.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

bkai_model = SentenceTransformer(BKVEC_MODEL, device=device)
clip_model = SentenceTransformer(IMG_MODEL, device=device)
clip_finetuned_model = SentenceTransformer(str(IMG_MODEL_FINETUNED), device=device)

print("BKAI dim:", bkai_model.get_sentence_embedding_dimension())
print("CLIP dim:", clip_model.get_sentence_embedding_dimension())
print("Fine-tuned CLIP dim:", clip_finetuned_model.get_sentence_embedding_dimension())

def embed_text_bkai(texts: list[str]) -> list[list[float]]:
    return bkai_model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True).tolist()

def embed_text_clip(texts: list[str], *, finetuned: bool = False) -> list[list[float]]:
    model = clip_finetuned_model if finetuned else clip_model
    return model.encode(texts, batch_size=16, normalize_embeddings=True, convert_to_numpy=True).tolist()

def sparse_vector(text: str) -> SparseVector:
    counts = {}
    for token in TOKEN_RE.findall(str(text or "").lower()):
        digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
        idx = int.from_bytes(digest, "big") % 2_147_483_647
        counts[idx] = counts.get(idx, 0) + 1
    indices = sorted(counts)
    values = [1.0 + math.log(counts[idx]) for idx in indices]
    norm = math.sqrt(sum(v * v for v in values)) or 1.0
    return SparseVector(indices=indices, values=[v / norm for v in values])


## Candidate Generation

Each branch searches one representation. Later cells fuse and rerank the candidates.

In [ ]:
def modality_filter(modality: str) -> models.Filter:
    return models.Filter(
        must=[models.FieldCondition(key="modality", match=models.MatchValue(value=modality))]
    )

def query_named_vector(collection: str, query_vector: Any, vector_name: str, modality: str, limit: int):
    return client.query_points(
        collection_name=collection,
        query=query_vector,
        using=vector_name,
        query_filter=modality_filter(modality),
        limit=limit,
        with_payload=True,
        with_vectors=False,
    ).points

def generate_candidates(pack: dict[str, Any], collection: str = DEFAULT_COLLECTION, per_branch: int = CANDIDATES_PER_BRANCH):
    branch_results: dict[str, list] = {}

    text_queries = pack["text_queries"][:6]
    if text_queries:
        vectors = embed_text_bkai(text_queries)
        hits = []
        for query_text, vector in zip(text_queries, vectors):
            points = query_named_vector(collection, vector, TEXT_VECTOR, "text", per_branch)
            hits.extend({"point": point, "query": query_text} for point in points)
        branch_results["text_dense"] = hits

    keyword_text = " ".join(pack["keyword_queries"])
    if keyword_text.strip():
        points = query_named_vector(collection, sparse_vector(keyword_text), SPARSE_VECTOR, "text", per_branch)
        branch_results["text_sparse"] = [{"point": point, "query": keyword_text} for point in points]

    visual_queries = pack["visual_queries"][:6]
    if visual_queries:
        original_vectors = embed_text_clip(visual_queries, finetuned=False)
        finetuned_vectors = embed_text_clip(visual_queries, finetuned=True)

        original_hits = []
        for query_text, vector in zip(visual_queries, original_vectors):
            points = query_named_vector(collection, vector, IMAGE_VECTOR, "image", per_branch)
            original_hits.extend({"point": point, "query": query_text} for point in points)
        branch_results["image_clip"] = original_hits

        finetuned_hits = []
        for query_text, vector in zip(visual_queries, finetuned_vectors):
            points = query_named_vector(collection, vector, IMAGE_VECTOR_FINETUNED, "image", per_branch)
            finetuned_hits.extend({"point": point, "query": query_text} for point in points)
        branch_results["image_clip_finetuned"] = finetuned_hits

    return branch_results

branches = generate_candidates(sample_pack, collection=DEFAULT_COLLECTION, per_branch=10)
{name: len(points) for name, points in branches.items()}


## Fusion And Reranking

Weighted RRF is robust because scores from dense, sparse, and CLIP branches are not directly comparable.

In [ ]:
BRANCH_WEIGHTS = {
    "text_dense": 1.20,
    "text_sparse": 1.00,
    "image_clip": 0.85,
    "image_clip_finetuned": 1.25,
}

cross_encoder = None
if CROSS_ENCODER_MODEL:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=device, max_length=CROSS_ENCODER_MAX_LENGTH)
    print(f"Loaded CrossEncoder reranker: {CROSS_ENCODER_MODEL}")


def point_key(point) -> str:
    return str(point.id)

def weighted_rrf(branch_results: dict[str, list], rrf_k: int = RRF_K) -> list[dict[str, Any]]:
    fused: dict[str, dict[str, Any]] = {}
    for branch, points in branch_results.items():
        seen_in_branch = set()
        for rank, record in enumerate(points, start=1):
            point = record["point"] if isinstance(record, dict) else record
            retrieval_query = record.get("query", "") if isinstance(record, dict) else ""
            key = point_key(point)
            if key in seen_in_branch:
                continue
            seen_in_branch.add(key)
            item = fused.setdefault(key, {
                "id": key,
                "payload": point.payload or {},
                "point": point,
                "rrf_score": 0.0,
                "branches": [],
                "best_raw_score": float(point.score),
            })
            item["rrf_score"] += BRANCH_WEIGHTS.get(branch, 1.0) / (rrf_k + rank)
            item["branches"].append({"branch": branch, "rank": rank, "score": float(point.score), "query": retrieval_query})
            item["best_raw_score"] = max(item["best_raw_score"], float(point.score))
    return sorted(fused.values(), key=lambda x: x["rrf_score"], reverse=True)

def text_overlap_score(query_pack: dict[str, Any], payload: dict[str, Any]) -> float:
    query_text = " ".join(query_pack["text_queries"] + query_pack["keyword_queries"] + query_pack["visual_queries"]).lower()
    payload_text = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "url", "image_path"]).lower()
    query_tokens = set(TOKEN_RE.findall(query_text))
    if not query_tokens:
        return 0.0
    hit_tokens = [tok for tok in query_tokens if tok in payload_text]
    return min(len(hit_tokens) / max(len(query_tokens), 1), 0.35)

def cross_encoder_boost(query_pack: dict[str, Any], payload: dict[str, Any]) -> float:
    if cross_encoder is None or payload.get("modality") != "text":
        return 0.0
    query = query_pack["text_queries"][0] if query_pack["text_queries"] else str(query_pack.get("claim", ""))
    passage = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "date"])
    if not passage.strip():
        return 0.0
    raw_score = float(cross_encoder.predict([(query, passage[:3000])])[0])
    normalized_score = 1.0 / (1.0 + math.exp(-raw_score))
    return normalized_score * 0.25

def heuristic_rerank(fused_items: list[dict[str, Any]], query_pack: dict[str, Any]) -> list[dict[str, Any]]:
    for item in fused_items:
        payload = item["payload"]
        branch_names = {b["branch"] for b in item["branches"]}
        branch_agreement = min(len(branch_names) * 0.025, 0.10)
        modality = payload.get("modality", "")
        modality_bonus = 0.03 if modality == "image" and query_pack["retrieval_focus"].get("image", False) else 0.0
        overlap = text_overlap_score(query_pack, payload)
        reranker_boost = cross_encoder_boost(query_pack, payload)
        item["final_score"] = item["rrf_score"] + branch_agreement + modality_bonus + overlap + reranker_boost
        item["reranker_boost"] = reranker_boost
    return sorted(fused_items, key=lambda x: x["final_score"], reverse=True)

fused = weighted_rrf(branches)
ranked = heuristic_rerank(fused, sample_pack)
pd.DataFrame([{
    "rank": i + 1,
    "modality": item["payload"].get("modality"),
    "final_score": item["final_score"],
    "rrf_score": item["rrf_score"],
    "branches": ", ".join(sorted({b["branch"] for b in item["branches"]})),
    "title": item["payload"].get("title", ""),
} for i, item in enumerate(ranked[:10])])


## Display Top 3 Evidence

This rendering cell is intended for manual inspection before sending evidence to the LLM judge.

In [ ]:
def resolve_image_path(image_path: str) -> Path:
    path = Path(str(image_path))
    if path.exists():
        return path
    return DATASET_ROOT / path

def image_data_uri(image_path: str, max_size: tuple[int, int] = (280, 190)) -> str | None:
    path = resolve_image_path(image_path)
    if not path.exists():
        return None
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            img.thumbnail(max_size)
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=85)
        return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")
    except Exception:
        return None

def render_evidence_card(rank: int, item: dict[str, Any]) -> str:
    payload = item["payload"]
    modality = payload.get("modality", "")
    branches = ", ".join(sorted({b["branch"] for b in item["branches"]}))
    title = html.escape(str(payload.get("title", ""))[:120])
    source = html.escape(str(payload.get("source", ""))[:100])
    url = html.escape(str(payload.get("url", ""))[:160])
    if modality == "image":
        image_path = str(payload.get("image_path", ""))
        data_uri = image_data_uri(image_path)
        media = f'<img src="{data_uri}" style="width:100%;height:180px;object-fit:contain;background:#f7f7f7;border:1px solid #ddd;" />' if data_uri else '<div style="height:180px;background:#f7f7f7;border:1px solid #ddd;display:flex;align-items:center;justify-content:center;color:#777;">missing image</div>'
        body = f'<code style="font-size:11px;word-break:break-word;white-space:normal;">{html.escape(image_path)}</code>'
    else:
        snippet = html.escape(str(payload.get("text", ""))[:700])
        media = '<div style="height:180px;background:#f7f7f7;border:1px solid #ddd;padding:10px;overflow:auto;font:13px/1.45 sans-serif;">' + snippet + '</div>'
        body = ""
    return f"""
    <div style="border:1px solid #d6d6d6;border-radius:6px;padding:10px;max-width:320px;min-width:260px;">
      <div style="font:600 14px sans-serif;margin-bottom:6px;">#{rank} {html.escape(str(modality))} evidence</div>
      {media}
      <div style="font:13px/1.45 sans-serif;margin-top:8px;">
        <b>final</b>={item['final_score']:.4f} | <b>rrf</b>={item['rrf_score']:.4f}<br>
        <b>branches</b>: {html.escape(branches)}<br>
        <b>title</b>: {title}<br>
        <b>source</b>: {source}<br>
        <b>url</b>: <span style="word-break:break-word;">{url}</span><br>
        {body}
      </div>
    </div>
    """

def display_top_evidence(pack: dict[str, Any], ranked_items: list[dict[str, Any]], top_k: int = FINAL_TOP_K):
    cards = [render_evidence_card(i + 1, item) for i, item in enumerate(ranked_items[:top_k])]
    display(HTML(f"""
    <div style="font-family:sans-serif;margin-bottom:10px;">
      <h3 style="margin:0 0 6px;">Top {top_k} evidence</h3>
      <div><b>Claim:</b> {html.escape(str(pack['claim']))}</div>
      <div><b>Refiner:</b> {html.escape(str(pack['model_alias']))}</div>
    </div>
    <div style="display:flex;gap:12px;flex-wrap:wrap;align-items:flex-start;">{''.join(cards)}</div>
    """))

display_top_evidence(sample_pack, ranked, top_k=FINAL_TOP_K)


## One Function For Phase 3

Use this function to retrieve top-3 evidence for one refined row. The return value is JSON-friendly for the LLM judge.

In [ ]:
def retrieve_top3_evidence(row: pd.Series, collection: str = DEFAULT_COLLECTION, *, display_results: bool = True) -> list[dict[str, Any]]:
    pack = build_query_pack(row)
    branches = generate_candidates(pack, collection=collection, per_branch=CANDIDATES_PER_BRANCH)
    fused = weighted_rrf(branches)
    ranked = heuristic_rerank(fused, pack)
    if display_results:
        display_top_evidence(pack, ranked, top_k=FINAL_TOP_K)
    evidence = []
    for rank, item in enumerate(ranked[:FINAL_TOP_K], start=1):
        payload = item["payload"]
        evidence.append({
            "rank": rank,
            "point_id": item["id"],
            "modality": payload.get("modality"),
            "final_score": item["final_score"],
            "rrf_score": item["rrf_score"],
            "branches": item["branches"],
            "title": payload.get("title", ""),
            "source": payload.get("source", ""),
            "url": payload.get("url", ""),
            "date": payload.get("date", ""),
            "text": payload.get("text", ""),
            "image_path": payload.get("image_path", ""),
            "corpus_id": payload.get("corpus_id"),
        })
    return evidence

# Example: choose by claim id and/or model alias.
row = pick_refined_row(claim_id=None, model_alias=None, index=0)
top3_evidence = retrieve_top3_evidence(row, collection=DEFAULT_COLLECTION, display_results=True)
top3_evidence


## Optional Next Steps

- Add OCR text for each image and index it into Qdrant as text evidence.
- Add a CrossEncoder text reranker for text candidates.
- Add a VLM relevance reranker for image candidates.
- Build a golden set and score Recall@3/10/20 before sending evidence to the LLM judge.